# 向量内积与距离

第九讲实验：从精确分数核验投影，再解释尺度如何改变近邻。

先阅读 lab.md，完成纸笔答案。全部向量与房屋记录为合成教学数据。数值代码只需 CPU、Python 标准库与 NumPy，不访问网络。逐格练习后，重启内核并运行全部。

## 1 记录环境

版本属于复现信息，不是数学定理的前提。

In [1]:
import sys
from fractions import Fraction
import numpy as np
print("Python:",sys.version.split()[0])
print("NumPy:",np.__version__)

Python: 3.12.14
NumPy: 2.3.5


## 2 向量与数组的输入输出

数学上二维向量有两个分量；本讲程序中它使用一维数组，shape 为 (2,)。

In [2]:
x=np.array([4.,1.])
y=np.array([3.,4.])
assert x.shape==y.shape==(2,)
print("相加:",x+y)
print("线性组合 2x-y:",2*x-y)
print("逐元素乘积:",x*y)
print("内积:",(x*y).sum(),np.dot(x,y))
assert np.dot(x,y)==16.

相加: [7. 5.]
线性组合 2x-y: [ 5. -2.]
逐元素乘积: [12.  4.]
内积: 16.0 16.0


## 3 精确有理数投影

从整数构造 Fraction。先检查系数，再检查投影与残差；不要先将近似小数当成精确分数。

In [3]:
xf=[Fraction(4),Fraction(1)]
yf=[Fraction(3),Fraction(4)]
inner=sum(a*b for a,b in zip(xf,yf))
y_squared=sum(v*v for v in yf)
af=inner/y_squared
pf=[af*v for v in yf]
rf=[a-b for a,b in zip(xf,pf)]
assert af==Fraction(16,25)
assert pf==[Fraction(48,25),Fraction(64,25)]
assert rf==[Fraction(52,25),Fraction(-39,25)]
assert sum(a*b for a,b in zip(rf,yf))==0
print("内积:",inner,"系数:",af)
print("投影:",[str(v) for v in pf],"残差:",[str(v) for v in rf])

内积: 16 系数: 16/25
投影: ['48/25', '64/25'] 残差: ['52/25', '-39/25']


## 4 精确平方长度分解

17 = 256/25 + 169/25。分解中的各项都是有理数，不需要对根号做近似才能核验。

In [4]:
xx=sum(v*v for v in xf)
pp=sum(v*v for v in pf)
rr=sum(v*v for v in rf)
assert xx==17 and pp==Fraction(256,25) and rr==Fraction(169,25)
assert xx==pp+rr
print("平方长度:",str(xx),"=",str(pp),"+",str(rr))

平方长度: 17 = 256/25 + 169/25


## 5 范数与平方范数

L1 为绝对值求和；L2 为平方求和再开平方。平方 L2 不是范数。

In [5]:
l1=np.abs(x).sum()
l2=np.sqrt((x*x).sum())
print("x 的 L1:",l1,"L2:",l2,"平方 L2:",(x*x).sum())
np.testing.assert_allclose(l2,np.linalg.norm(x,ord=2),rtol=0,atol=1e-12)
print("2x 的 L2:",np.linalg.norm(2*x),"平方 L2:",np.sum((2*x)**2))
assert l1==5.

x 的 L1: 5.0 L2: 4.123105625617661 平方 L2: 17.0
2x 的 L2: 8.246211251235321 平方 L2: 68.0


## 6 点之间的距离

x 与 z 的差向量为 (3,-4)。再用三个一维点显示平方距离为什么不满足三角不等式。

In [6]:
from experiment import distance,norm1,norm2
z=np.array([1.,5.])
print("差向量:",x-z)
print("L2 距离:",distance(x,z),"L1 距离:",distance(x,z,order=1))
assert distance(x,z)==5. and distance(x,z,order=1)==7.
left=distance([0],[2])**2
right=distance([0],[1])**2+distance([1],[2])**2
print("平方距离反例:",left,">",right)
assert left>right

差向量: [ 3. -4.]
L2 距离: 5.0 L1 距离: 7.0
平方距离反例: 4.0 > 2.0


## 7 浮点投影需要容差核验

与精确分数结果逐项比较。残差点积可能出现极小舍入误差，但不应出现明显非零结果。

In [7]:
from experiment import project_line,dot_checked
alpha,p,r=project_line(x,y)
assert p.shape==r.shape==(2,)
np.testing.assert_allclose(p,[float(v) for v in pf],rtol=0,atol=1e-12)
np.testing.assert_allclose(r,[float(v) for v in rf],rtol=0,atol=1e-12)
np.testing.assert_allclose(dot_checked(r,y),0.,rtol=0,atol=1e-12)
np.testing.assert_allclose(dot_checked(x,x),dot_checked(p,p)+dot_checked(r,r),rtol=0,atol=1e-12)
print("系数:",alpha,"投影:",p,"残差:",r)
print("残差与方向内积:",dot_checked(r,y))

系数: 0.64 投影: [1.92 2.56] 残差: [ 2.08 -1.56]
残差与方向内积: 0.0


## 8 最近点恒等式的有限核验

以下六个系数用于检查实现。完整结论来自讲稿对任意实数系数的非负平方推导，不来自只检查六个点。

In [8]:
for c in [-2.,0.,.5,alpha,1.,2.]:
    difference=x-c*y
    actual=dot_checked(difference,difference)
    expected=dot_checked(r,r)+(alpha-c)**2*dot_checked(y,y)
    np.testing.assert_allclose(actual,expected,rtol=0,atol=1e-12)
    assert actual>=dot_checked(r,r)-1e-12
    print("c=",c,"平方距离=",round(actual,8))

c= -2.0 平方距离= 181.0
c= 0.0 平方距离= 17.0
c= 0.5 平方距离= 7.25
c= 0.64 平方距离= 6.76
c= 1.0 平方距离= 10.0
c= 2.0 平方距离= 53.0


## 9 改变方向长度 不改变整条直线

方向换成正倍数或负倍数，投影点不变；系数根据方向表示改变。

In [9]:
for factor in [2.,-3.]:
    a2,p2,r2=project_line(x,factor*y)
    np.testing.assert_allclose(p2,p,rtol=0,atol=1e-12)
    np.testing.assert_allclose(a2,alpha/factor,rtol=0,atol=1e-12)
    print("方向倍数:",factor,"新系数:",a2,"同一投影点:",p2)
print("零来源的投影:",project_line([0.,0.],y)[1])

方向倍数: 2.0 新系数: 0.32 同一投影点: [1.92 2.56]
方向倍数: -3.0 新系数: -0.21333333333333335 同一投影点: [1.92 2.56]
零来源的投影: [0. 0.]


## 10 单位方向 余弦 与角度

arccos 返回弧度，degrees 转为度。主例的夹角约 39.0939 度，不是有方向的顺逆时针转角。

In [10]:
from experiment import unit_direction,cosine_checked,angle_degrees
u=unit_direction(y)
cosine=cosine_checked(x,y)
print("y 的单位方向:",u)
print("归一化内积:",cosine)
print("弧度:",np.arccos(cosine),"角度:",angle_degrees(x,y))
np.testing.assert_allclose(u,[.6,.8],rtol=0,atol=1e-12)
print("三个基准角:",[angle_degrees([1,0],v) for v in [[1,0],[0,1],[-1,0]]])

y 的单位方向: [0.6 0.8]
归一化内积: 0.7761140001162654
弧度: 0.6823165548747483 角度: 39.09385888622951
三个基准角: [0.0, 90.0, 180.0]


## 11 零向量按运算分别处理

零向量长度合法；单位方向、余弦、角度没有定义；零投影方向被本接口拒绝。

In [11]:
from experiment import expect_error
print("零向量长度:",norm2([0,0]))
for name,operation in [
    ("零方向投影",lambda:project_line(x,[0,0])),
    ("零向量归一化",lambda:unit_direction([0,0])),
    ("零向量余弦",lambda:cosine_checked([0,0],y)),
    ("零向量角度",lambda:angle_degrees(x,[0,0])),
]:
    expect_error(operation)
    print("已按定义域拒绝:",name)

零向量长度: 0.0
已按定义域拒绝: 零方向投影
已按定义域拒绝: 零向量归一化
已按定义域拒绝: 零向量余弦
已按定义域拒绝: 零向量角度


## 12 夹取只容忍极小舍入

不要用 clip 掩盖明显错误。1.01 不能被当成一个近似有效余弦。

In [12]:
from experiment import clip_roundoff_cosine
print("极小上界舍入:",clip_roundoff_cosine(1+5e-13))
print("极小下界舍入:",clip_roundoff_cosine(-1-5e-13))
expect_error(lambda:clip_roundoff_cosine(1.01))
expect_error(lambda:clip_roundoff_cosine(np.nan))
print("明显越界与 nan 均已拒绝")

极小上界舍入: 1.0
极小下界舍入: -1.0
明显越界与 nan 均已拒绝


## 13 三行两列的批量计算

每行是一条二维向量。每个样本一个内积；每个样本的投影仍有两个分量。

In [13]:
from experiment import row_dots,row_projections,row_distances
X=np.array([[4.,1.],[1.,-2.],[0.,0.]])
before=X.copy()
dots=row_dots(X,y)
coefficients,projected,residuals=row_projections(X,y)
assert dots.shape==coefficients.shape==(3,)
assert projected.shape==residuals.shape==(3,2)
np.testing.assert_allclose(dots,[16,-5,0],rtol=0,atol=1e-12)
np.testing.assert_allclose(projected,np.array([project_line(row,y)[1] for row in X]),rtol=0,atol=1e-12)
np.testing.assert_array_equal(X,before)
print("批量内积:",dots)
print("投影与残差形状:",projected.shape,residuals.shape)
print("每行 L2 长度:",np.linalg.norm(X,ord=2,axis=1))

批量内积: [16. -5.  0.]
投影与残差形状: (3, 2) (3, 2)
每行 L2 长度: [4.12310563 2.23606798 0.        ]


## 14 单样本 单特征 与拒绝边界

保留样本轴，单样本也不变成一个无批量轴的向量。

In [14]:
assert row_dots(X[:1],y).shape==(1,)
assert row_projections(X[:1],y)[1].shape==(1,2)
np.testing.assert_array_equal(row_dots(np.array([[2.],[3.],[4.]]),[5.]),[10,15,20])
for operation in [lambda:row_dots(X[0],y),lambda:dot_checked([1,2],[1]),
                  lambda:dot_checked([1+2j],[1]),lambda:dot_checked([True,1],[1,2]),lambda:row_dots(np.empty((0,2)),y)]:
    expect_error(operation)
print("单样本、单特征通过；错误轴、错误长度、复数、混合布尔输入、空批量已拒绝")

单样本、单特征通过；错误轴、错误长度、复数、混合布尔输入、空批量已拒绝


## 15 范数选择也能翻转近邻

查询为原点，候选 (3,0) 与 (2,2)。这里没有换单位，改变的是范数定义。

In [15]:
for order in [1,2]:
    values=[distance([3,0],[0,0],order=order),distance([2,2],[0,0],order=order)]
    print("L",order,"距离:",values,"更近:",["A","B"][int(np.argmin(values))])

L 1 距离: [3.0, 4.0] 更近: A
L 2 距离: [3.0, 2.8284271247461903] 更近: B


## 16 原始坐标换单位

Q=(50平方米,5年)，A=(51平方米,15年)，B=(56平方米,6年)。下面只是混合单位数字的欧氏距离，不具有统一物理长度单位。

In [16]:
Q=np.array([50.,5.])
candidates=np.array([[51.,15.],[56.,6.]])
factor=np.array([100.,1.])
raw=row_distances(candidates,Q)
converted=row_distances(candidates*factor,Q*factor)
print("平方米坐标原始距离:",raw,"更近:",["A","B"][int(np.argmin(raw))])
print("平方分米坐标原始距离:",converted,"更近:",["A","B"][int(np.argmin(converted))])
assert int(np.argmin(raw))==1 and int(np.argmin(converted))==0

平方米坐标原始距离: [10.04987562  6.08276253] 更近: B
平方分米坐标原始距离: [100.49875621 600.00083333] 更近: A


## 17 同步转换参考尺度

面积参考 10 平方米，楼龄参考 5 年。面积换单位时参考改为 1000 平方分米；无量纲距离应保持不变。

In [17]:
scales=np.array([10.,5.])
old=row_distances(candidates,Q,scales)
new=row_distances(candidates*factor,Q*factor,scales*factor)
np.testing.assert_allclose(old,new,rtol=0,atol=1e-12)
print("无量纲距离:",old,"同步换单位后:",new)
expect_error(lambda:row_distances(candidates,Q,[0.,5.]))
expect_error(lambda:row_distances(candidates,Q,[-10.,5.]))
expect_error(lambda:row_distances(candidates,Q,[10.]))
print("零尺度、负尺度与错误长度已拒绝")

无量纲距离: [2.00249844 0.63245553] 同步换单位后: [2.00249844 0.63245553]
零尺度、负尺度与错误长度已拒绝


## 18 极端浮点范围的边界

很小不等于数学上的零。单位方向先缩放可避免过早下溢；直接投影分母下溢和大数点积溢出会明确报错。

In [18]:
print("很小非零向量的方向:",unit_direction([1e-300,0]))
expect_error(lambda:project_line(x,[1e-300,0]),ArithmeticError)
expect_error(lambda:dot_checked([1e308],[1e308]),FloatingPointError)
print("数值下溢与溢出边界已经报告，不将它们伪装成普通结果")

很小非零向量的方向: [1. 0.]
数值下溢与溢出边界已经报告，不将它们伪装成普通结果


## 19 完整验证 不冒充数学证明

脚本包含精确手算、边界、非方阵、单样本、尺度变换与 120 组有限随机核验。随机检查发现实现问题，讲稿的证明负责一般结论。

In [19]:
from experiment import run_checks
report=run_checks()
assert report["status"]=="passed" and len(report["checks"])==17
print("状态:",report["status"],"检查组数:",len(report["checks"]))
for check in report["checks"]:
    print(check["name"],check["passed"])
print("精确分数:",report["exact_fraction_results"])
print("主要数值:",report["numeric_results"])

状态: passed 检查组数: 17
exact_fraction_projection_and_energy_identity True
vector_arithmetic_and_dot_properties True
norms_and_known_distances True
float_projection_residual_orthogonality_and_energy True
nearest_point_identity_on_six_coefficients True
direction_rescaling_and_zero_source_projection True
zero_direction_normalization_and_angle_rejected True
unit_directions_angles_and_guarded_roundoff_clipping True
non_square_batch_matches_loop_without_input_mutation True
single_sample_and_single_feature_shapes True
shape_empty_real_finite_and_scale_contracts True
raw_nearest_neighbor_flip_and_dimensionless_invariance True
l1_l2_choice_can_reverse_nearest_candidate True
120_finite_random_checks_of_bounds_and_distance_not_a_proof True
equality_cases_and_squared_distance_triangle_counterexample True
translation_preserves_distance_but_not_origin_based_cosine True
extreme_float_limits_are_reported_not_silently_clipped True
精确分数: {'dot': '16', 'coefficient': '16/25', 'projection': ['48/25', '64/25'

## 实验记录

在新文字格回答：

1. 投影的唯一最近点证明使用了哪些条件？零方向为什么需要独立处理？
2. 本次近邻比较采用什么坐标、单位与参考尺度？哪些改变只是换单位，哪些改变了定义？
3. 数值检查通过后，还不能自动宣称哪些数学或业务结论？

最后重启内核并运行全部，保存输出。十二道纸笔练习与完整详解见 lecture.md。